# 🌊 Évaluation Multi-Modèles LSTM Régionaux sur les Nouvelles Stations du Maroc
### Bassins de l'Oum Er Rbia (ABHOER) et du Loukkos (ABHL)
**Projet de Fin d'Études — IAV Hassan II**

Ce notebook exécute l'évaluation rigoureuse en conditions réelles non jaugées (**PUB - Prediction in Ungauged Basins**) sur 9 stations hydrométriques marocaines :
- **5 stations dans l'Oum Er Rbia :** Addammaghène, Sgatt, Zaouit Ahançal, Tillouguite, Tizi n'Isly.
- **4 stations dans le Loukkos :** Pont d'Oughane, Pont M'Ghar, M'Douar, Boufarah.

---

### 🔬 Modèles Évalués et Comparés :
1. **Modèle Temporel de Référence (`temporal_model`) :** Entraîné sur **247 bassins** espagnols pendant 25 époques (Score d'entraînement de référence : **KGE médian = 0,619**).
2. **Modèle de Déploiement Graine 42 (`deploy_s42`) :** Entraîné sur 247 bassins pendant 20 époques.
3. **Modèle de Déploiement Graine 1042 (`deploy_s1042`) :** Graine stochastique 1042.
4. **Modèle de Déploiement Graine 2042 (`deploy_s2042`) :** Graine stochastique 2042.
5. **Ensemble Multi-Graines (`ensemble_3seeds`) :** Moyenne d'ensemble $\bar{Q} = \frac{Q_{42} + Q_{1042} + Q_{2042}}{3}$ (réduction de variance stochastique).

---

### 💡 Atout Majeur du PFE :
Toutes ces stations ont des superficies comprises entre **82 km² et 2 501 km²**, ce qui les place **strictement à l'intérieur de l'enveloppe de validité opérationnelle** du modèle ($20 \le \text{Aire} \le 3500\text{ km}^2$) !


In [ ]:
# 1. Dépendances et Connexion Google Drive
!pip -q install earthengine-api geopandas pyyaml torch rasterio shapely matplotlib seaborn

import os, sys, glob, time, calendar
import numpy as np
import pandas as pd
import geopandas as gpd
import yaml
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Montage Google Drive
if os.path.exists('/content/drive'):
    print("Google Drive déjà connecté.")
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        print("Exécution en local (hors Google Colab).")


## 1. Initialisation de Google Earth Engine
Connexion sécurisée via votre projet Google Cloud Earth Engine (`pfe-rainfall`).


In [ ]:
import ee

# Projet Google Cloud Earth Engine configuré pour le PFE
PROJECT_ID = 'pfe-rainfall'

try:
    ee.Initialize(project=PROJECT_ID)
    print(f"Earth Engine initialisé avec succès sur le projet : {PROJECT_ID}")
except Exception as e:
    print("Initialisation directe échouée, tentative d'authentification...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print(f"Earth Engine authentifié et initialisé sur le projet : {PROJECT_ID}")


## 2. Chargement des Données & des 9 Bassins Versants Délinéés
Chargement des polygones vectoriels et des séries mensuelles observées des agences de bassin.


In [ ]:
# Chemins automatiques (Google Drive ou Local)
POSSIBLE_DATA_DIRS = [
    "/content/drive/MyDrive/pfe_rainfall/morocco",
    "./data"
]

POSSIBLE_MODELS_DIRS = [
    "/content/drive/MyDrive/pfe_rainfall/models",
    "./models"
]

DATA_DIR = next((p for p in POSSIBLE_DATA_DIRS if os.path.exists(p)), ".")
MODELS_DIR = next((p for p in POSSIBLE_MODELS_DIRS if os.path.exists(p)), ".")

print(f"DATA_DIR   : {DATA_DIR}")
print(f"MODELS_DIR : {MODELS_DIR}")

# 1. Chargement des bassins
gpkg_path = os.path.join(DATA_DIR, "catchments_new_stations.gpkg")
cat = gpd.read_file(gpkg_path, layer="catchments")
print(f"\n--- {len(cat)} Bassins Versants Délinéés ---")
print(cat[["code", "name", "basin", "river", "delineated_area_km2", "target_area_km2"]].to_string(index=False))

# 2. Chargement des débits observés
obs_path = os.path.join(DATA_DIR, "observed_monthly_9_stations.csv")
df_obs = pd.read_csv(obs_path)
df_obs["date"] = pd.to_datetime(df_obs["date"])
print(f"\nChargé {len(df_obs)} enregistrements mensuels observés (de {df_obs['date'].min().strftime('%Y-%m')} à {df_obs['date'].max().strftime('%Y-%m')}).")


## 3. Extraction des Attributs Physiques (Topographie & Occupation du sol)
Extraction zonale via Earth Engine :
- `elev_mean` : Altitude moyenne (MERIT Hydro)
- `slope_mean` : Pente moyenne (MERIT Hydro)
- `forest_frac` : Fraction forestière (ESA WorldCover 10m)


In [ ]:
cat_s = cat.copy()
cat_s["geometry"] = cat_s.geometry.simplify(0.002)

fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry(r.geometry.__geo_interface__), {"code": r["code"], "name": r["name"]})
    for _, r in cat_s.iterrows()
])

# 1. Topographie (MERIT Hydro)
elev = ee.Image("MERIT/Hydro/v1_0_1").select("elv").rename("elev_mean")
slope = ee.Terrain.slope(elev).rename("slope_mean")

# 2. Occupation du sol (ESA WorldCover)
forest = ee.ImageCollection("ESA/WorldCover/v200").first().eq(10).rename("forest_frac")

# Réduction zonale
topo_img = elev.addBands([slope, forest])
reduced = topo_img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=250).map(lambda f: f.setGeometry(None))
df_topo = pd.DataFrame([d["properties"] for d in reduced.getInfo()["features"]])

df_attributes = cat[["code", "name", "basin", "delineated_area_km2"]].rename(columns={"delineated_area_km2": "area_km2"}).merge(df_topo, on=["code", "name"])
print("=== ATTRIBUTS PHYSIQUES EXTRAITS ===")
print(df_attributes.to_string(index=False))


## 4. Extraction des Forçages Météorologiques ERA5-Land (2000 - 2022)
Calcul de l'ETP Hargreaves et des 4 attributs climatiques :
- `p_mean` : Précipitation moyenne journalière (mm/jour)
- `aridity` : Indice d'aridité ($ET_0 / P$)
- `frac_snow` : Fraction des précipitations sous forme de neige
- `p_seasonality` : Indice de saisonnalité des pluies


In [ ]:
# 4. Extraction ERA5-Land Ultra-Rapide (Parallélisation sur les 9 stations via reduceRegions)
def hargreaves_daily(dates, tmin, tmax, tmean, lat):
    J = pd.to_datetime(dates).dt.dayofyear.values
    phi = np.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * J / 365.0)
    dec = 0.409 * np.sin(2 * np.pi * J / 365.0 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    Ra = (24 * 60 / np.pi) * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(ws))
    return np.clip(0.0023 * (Ra * 0.408) * (tmean + 17.8) * np.sqrt(np.clip(tmax - tmin, 0, None)), 0, None)

cache_file = os.path.join(DATA_DIR, "forcings_9_stations_era5.parquet")
forcings_dict = {}

if os.path.exists(cache_file):
    print(f"Chargement direct des forçages depuis le cache Drive : {cache_file}")
    df_cache = pd.read_parquet(cache_file)
    for code, g in df_cache.groupby("code"):
        forcings_dict[code] = g.sort_values("date").reset_index(drop=True)
    print(f"-> {len(forcings_dict)} stations chargées instantanément depuis le cache !")
else:
    print(f"Extraction parallèle ERA5-Land pour les {len(cat)} stations (2000-2022)...")
    era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").select(
        ["total_precipitation_sum", "temperature_2m_min", "temperature_2m_max"]
    )
    
    # FeatureCollection contenant les 9 bassins
    fc_stations = ee.FeatureCollection([
        ee.Feature(ee.Geometry(r.geometry.__geo_interface__), {"code": r["code"], "name": r["name"], "lat": r["lat_snap"]})
        for _, r in cat_s.iterrows()
    ])
    
    START_YEAR = 2000
    END_YEAR = 2022
    total_years = END_YEAR - START_YEAR + 1
    
    station_data = {r["code"]: [] for _, r in cat.iterrows()}
    
    print(f"Lancement de l'extraction annuelle (1 requête GEE pour les 9 stations à la fois)...")
    for y_idx, year in enumerate(range(START_YEAR, END_YEAR + 1)):
        t_start = time.time()
        col_year = era5.filterDate(f"{year}-01-01", f"{year}-12-31")
        img_multi = col_year.toBands()
        
        # reduceRegions extrait les 9 bassins en un seul appel parallèle !
        res = img_multi.reduceRegions(
            collection=fc_stations,
            reducer=ee.Reducer.mean(),
            scale=2500,
            tileScale=4
        ).getInfo()
        
        for feat in res["features"]:
            code = feat["properties"]["code"]
            for key, val in feat["properties"].items():
                if key not in ["code", "name", "lat"]:
                    parts = key.split('_')
                    if len(parts) >= 2 and val is not None:
                        date_str = parts[0]
                        var_name = "_".join(parts[1:])
                        station_data[code].append({"date_raw": date_str, "var": var_name, "val": val})
                        
        dt = time.time() - t_start
        print(f"[{y_idx+1}/{total_years}] Année {year} extraite pour les 9 stations en {dt:.1f}s", flush=True)
        
    # Structuration en DataFrames journaliers
    all_stations_dfs = []
    for idx, r in cat.iterrows():
        code = r["code"]
        records = station_data[code]
        if not records:
            print(f"Avertissement : pas de données pour {code}")
            continue
            
        df_raw = pd.DataFrame(records)
        df_st = df_raw.pivot(index="date_raw", columns="var", values="val").reset_index()
        df_st["date"] = pd.to_datetime(df_st["date_raw"], format="%Y%m%d")
        df_daily = df_st.sort_values("date").copy()
        
        # Noms standardisés
        df_daily["prcp_mm"] = df_daily["total_precipitation_sum"] * 1000.0
        df_daily["tmin_c"] = df_daily["temperature_2m_min"] - 273.15
        df_daily["tmax_c"] = df_daily["temperature_2m_max"] - 273.15
        df_daily["temp_mean"] = (df_daily["tmin_c"] + df_daily["tmax_c"]) / 2.0
        df_daily["pet_mm"] = hargreaves_daily(df_daily["date"], df_daily["tmin_c"], df_daily["tmax_c"], df_daily["temp_mean"], r["lat_snap"])
        
        df_daily["precipitation"] = df_daily["prcp_mm"]
        df_daily["temp_min"] = df_daily["tmin_c"]
        df_daily["temp_max"] = df_daily["tmax_c"]
        df_daily["pet"] = df_daily["pet_mm"]
        df_daily["code"] = code
        
        forcings_dict[code] = df_daily
        all_stations_dfs.append(df_daily)
        
    if all_stations_dfs:
        df_all_forc = pd.concat(all_stations_dfs, ignore_index=True)
        try:
            df_all_forc.to_parquet(cache_file, index=False)
            print(f"\nForçages sauvegardés dans le cache Drive : {cache_file}")
        except Exception as e:
            print(f"Info cache : {e}")

# Calcul des 4 attributs climatiques
clim_rows = []
for code, df_daily in forcings_dict.items():
    p = df_daily["prcp_mm"].values
    tm = df_daily["temp_mean"].values
    pet = df_daily["pet_mm"].values
    pm = np.nanmean(p)
    mon_means = df_daily.groupby(df_daily["date"].dt.month)["prcp_mm"].mean()
    
    clim_rows.append({
        "code": code,
        "p_mean": round(float(pm), 3),
        "pet_mean": round(float(np.nanmean(pet)), 3),
        "aridity": round(float(np.nanmean(pet) / pm), 3),
        "frac_snow": round(float(np.nansum(p[tm < 0]) / np.nansum(p)), 4),
        "p_seasonality": round(float((mon_means.max() - mon_means.min()) / pm), 3)
    })

df_clim = pd.DataFrame(clim_rows)
df_full_attr = df_attributes.merge(df_clim, on="code")
print(f"\n=== MATRICE COMPLÈTE DES 8 ATTRIBUTS ({len(df_full_attr)} STATIONS) ===")
print(df_full_attr.to_string(index=False))


## 5. Architecture PyTorch & Chargement des Modèles
Définition du modèle LSTM régional standard de NeuralHydrology et chargement de tous les checkpoints et scalers.


In [ ]:
# 5. Architecture PyTorch & Chargement des Modèles
class RegressionHead(nn.Module):
    def __init__(self, in_features, out_features=1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_features, out_features))
    def forward(self, x):
        return self.net(x)

class NeuralHydrologyLSTM(nn.Module):
    def __init__(self, input_dim=12, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = RegressionHead(hidden_dim, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_step = lstm_out[:, -1, :]
        return self.head(self.dropout(last_step))

MODELS_CONFIG = {
    "temporal_model": {
        "dir": os.path.join(MODELS_DIR, "temporal_model"),
        "weights": "model_epoch025.pt",
        "name": "Modèle Temporel (247 bassins, Ep 25 - KGE ref: 0.619)"
    },
    "deploy_s42": {
        "dir": os.path.join(MODELS_DIR, "deploy_s42"),
        "weights": "model_epoch020.pt",
        "name": "Déploiement Graine 42 (Ep 20)"
    },
    "deploy_s1042": {
        "dir": os.path.join(MODELS_DIR, "deploy_s1042"),
        "weights": "model_epoch020.pt",
        "name": "Déploiement Graine 1042 (Ep 20)"
    },
    "deploy_s2042": {
        "dir": os.path.join(MODELS_DIR, "deploy_s2042"),
        "weights": "model_epoch020.pt",
        "name": "Déploiement Graine 2042 (Ep 20)"
    }
}

loaded_models = {}
loaded_scalers = {}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Calculs sur le device : {device}")

for m_key, m_info in MODELS_CONFIG.items():
    weights_path = os.path.join(m_info["dir"], m_info["weights"])
    scaler_path = os.path.join(m_info["dir"], "train_data_scaler.yml")
    
    if os.path.exists(weights_path) and os.path.exists(scaler_path):
        with open(scaler_path, "r") as f:
            sc = yaml.safe_load(f)
        loaded_scalers[m_key] = sc
        
        model = NeuralHydrologyLSTM(input_dim=12, hidden_dim=128, dropout=0.4).to(device)
        state_dict = torch.load(weights_path, map_location=device, weights_only=True)
        # Load state dict
        try:
            model.load_state_dict(state_dict, strict=True)
        except Exception:
            clean_sd = {k.replace("model.", ""): v for k, v in state_dict.items()}
            model.load_state_dict(clean_sd, strict=False)
            
        model.eval()
        loaded_models[m_key] = model
        print(f"Chargé avec succès : {m_info['name']}")
    else:
        print(f"MANQUANT : {m_key} à {m_info['dir']}")


## 6. Exécution de l'Inférence Journalière & Agrégation Mensuelle
Prédiction pour chaque modèle, calcul de l'Ensemble Multi-Graines, et conversion journalière ($\text{m}^3/\text{s}$) $\rightarrow$ mensuelle ($\text{m}^3/\text{s}$ et $\text{hm}^3/\text{mois}$).


In [ ]:
# 6. Exécution de l'Inférence Journalière (Ultra-Rapide & Sans dépassement mémoire GPU)
DYN_VARS = ['prcp_mm', 'tmin_c', 'tmax_c', 'pet_mm']
STATIC_ATTRS = ['area_km2', 'elev_mean', 'slope_mean', 'forest_frac', 'aridity', 'p_mean', 'frac_snow', 'p_seasonality']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"🚀 GPU ACTIVÉ : {torch.cuda.get_device_name(0)}")
else:
    print("ℹ️ Exécution sur CPU")

DYN_MAP = {
    'prcp_mm': 'prcp_mm' if 'prcp_mm' in list(forcings_dict.values())[0] else 'precipitation',
    'tmin_c': 'tmin_c' if 'tmin_c' in list(forcings_dict.values())[0] else 'temp_min',
    'tmax_c': 'tmax_c' if 'tmax_c' in list(forcings_dict.values())[0] else 'temp_max',
    'pet_mm': 'pet_mm' if 'pet_mm' in list(forcings_dict.values())[0] else 'pet'
}

SEQ_LEN = 365
BATCH_SIZE = 256  # Taille de lot standard NeuralHydrology (consomme < 200 Mo de VRAM GPU)
predictions = {}

t_start_total = time.time()

with torch.no_grad():
    for st_idx, (code, df_daily) in enumerate(forcings_dict.items()):
        t_st = time.time()
        st_row = df_full_attr[df_full_attr["code"] == code].iloc[0]
        area = st_row["area_km2"]
        N = len(df_daily)
        
        predictions[code] = {"dates": df_daily["date"].iloc[SEQ_LEN-1:].reset_index(drop=True)}
        
        for m_key, model in loaded_models.items():
            model.to(device)
            model.eval()
            sc = loaded_scalers[m_key]
            
            # 1. Normalisation dynamique vectorisée
            norm_dyn = []
            for d in DYN_VARS:
                center = sc['xarray_feature_center']['data_vars'][d]['data']
                scale = sc['xarray_feature_scale']['data_vars'][d]['data']
                col_name = DYN_MAP[d]
                s_norm = (df_daily[col_name].values - center) / scale
                norm_dyn.append(s_norm)
            norm_dyn = np.stack(norm_dyn, axis=1).astype(np.float32)
            
            # 2. Normalisation statique
            norm_statics = []
            for a in STATIC_ATTRS:
                mean = sc['attribute_means'][a]
                std = sc['attribute_stds'][a]
                val = (st_row[a] - mean) / std
                norm_statics.append(val)
            norm_statics = np.array(norm_statics, dtype=np.float32)
            
            # 3. Concaténation [4 dynamiques + 8 statiques]
            stat_matrix = np.tile(norm_statics, (N, 1))
            X_all = np.concatenate([norm_dyn, stat_matrix], axis=1)
            
            # 4. Fenêtrage glissant instantané (reste en RAM CPU)
            windows = np.lib.stride_tricks.sliding_window_view(X_all, (SEQ_LEN, 12))[:, 0, :, :].copy()
            X_tensor = torch.from_numpy(windows)
            
            # 5. Inférence GPU par mini-lots de 256 (ultra-économe en VRAM)
            outs = []
            for b_start in range(0, len(X_tensor), BATCH_SIZE):
                b_end = min(b_start + BATCH_SIZE, len(X_tensor))
                batch = X_tensor[b_start:b_end].to(device)
                pred = model(batch).cpu().numpy().squeeze()
                outs.append(pred)
                del batch
            
            out = np.concatenate(outs) if len(outs) > 1 else outs[0]
            
            # 6. Dénormalisation physique en m3/s
            q_center = sc['xarray_feature_center']['data_vars']['qobs_mm']['data']
            q_scale = sc['xarray_feature_scale']['data_vars']['qobs_mm']['data']
            q_mm = out * q_scale + q_center
            q_mm = np.clip(q_mm, 0.0, None)
            q_m3s = (q_mm * area * 1000.0) / 86400.0
            
            predictions[code][m_key] = q_m3s
            
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
        # 7. Calcul de l'Ensemble Multi-Graines (moyenne de s42, s1042, s2042)
        deploy_keys = [k for k in ["deploy_s42", "deploy_s1042", "deploy_s2042"] if k in loaded_models]
        if deploy_keys:
            predictions[code]["ensemble_3seeds"] = np.mean([predictions[code][k] for k in deploy_keys], axis=0)
            
        print(f"[{st_idx+1}/{len(forcings_dict)}] Station [{code}] {st_row['name']:<18} : {len(loaded_models)} modèles prédits en {time.time() - t_st:.2f}s", flush=True)

print(f"\n🎉 Inférence terminée avec succès pour toutes les stations en {time.time() - t_start_total:.2f} secondes !")


## 7. Agrégation Mensuelle & Métriques de Performance (KGE, NSE, $R^2$, PBIAS)
Comparaison directe avec les observations officielles des agences de bassin.


In [ ]:
def calc_metrics(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    o, s = obs[mask], sim[mask]
    if len(o) < 12:
        return {"KGE": np.nan, "NSE": np.nan, "R2": np.nan, "PBIAS": np.nan}
    
    # KGE
    r = np.corrcoef(o, s)[0, 1]
    alpha = np.std(s) / (np.std(o) + 1e-6)
    beta = np.mean(s) / (np.mean(o) + 1e-6)
    kge = 1.0 - np.sqrt((r - 1.0)**2 + (alpha - 1.0)**2 + (beta - 1.0)**2)
    
    # NSE
    nse = 1.0 - np.sum((o - s)**2) / (np.sum((o - np.mean(o))**2) + 1e-6)
    
    # R2
    r2 = r**2
    
    # PBIAS
    pbias = 100.0 * np.sum(s - o) / (np.sum(o) + 1e-6)
    
    return {"KGE": round(kge, 3), "NSE": round(nse, 3), "R2": round(r2, 3), "PBIAS": round(pbias, 1)}

# Agrégation mensuelle
results_summary = []

for code in predictions:
    st_meta = cat[cat["code"] == code].iloc[0]
    df_pred = pd.DataFrame({"date": predictions[code]["dates"]})
    
    # Modèles disponibles
    mod_keys = [k for k in predictions[code] if k != "dates"]
    for k in mod_keys:
        df_pred[k] = predictions[code][k]
        
    df_pred["ym"] = df_pred["date"].dt.to_period("M")
    df_pred_monthly = df_pred.groupby("ym")[mod_keys].mean().reset_index()
    df_pred_monthly["date"] = df_pred_monthly["ym"].dt.to_timestamp()
    
    # Join with observed
    st_obs = df_obs[df_obs["station_code"] == code].copy()
    merged = pd.merge(df_pred_monthly, st_obs[["date", "q_obs_m3s"]], on="date", how="inner")
    
    obs_vals = merged["q_obs_m3s"].values
    
    row_res = {
        "Code": code,
        "Station": st_meta["name"],
        "Bassin": st_meta["basin"],
        "Aire (km2)": st_meta["delineated_area_km2"],
        "N_Mois": len(merged)
    }
    
    for k in mod_keys:
        sim_vals = merged[k].values
        met = calc_metrics(obs_vals, sim_vals)
        row_res[f"KGE_{k}"] = met["KGE"]
        row_res[f"NSE_{k}"] = met["NSE"]
        
    results_summary.append(row_res)

df_results = pd.DataFrame(results_summary)
print("=== RÉSULTATS COMPARATIFS SUR LES 9 NOUVELLES STATIONS DU MAROC ===")
cols_show = ["Code", "Station", "Bassin", "Aire (km2)", "N_Mois"]
for k in ["temporal_model", "deploy_s42", "ensemble_3seeds"]:
    if f"KGE_{k}" in df_results.columns:
        cols_show.append(f"KGE_{k}")
print(df_results[cols_show].to_string(index=False))


## 8. Visualisation des Hydrogrammes Comparatifs
Tracé des débits mensuels observés vs prédits par le modèle temporel et l'ensemble de déploiement.


In [ ]:
fig, axes = plt.subplots(len(predictions), 1, figsize=(16, 3.5 * len(predictions)), sharex=False)
if len(predictions) == 1:
    axes = [axes]

for ax, code in zip(axes, predictions):
    st_meta = cat[cat["code"] == code].iloc[0]
    
    df_pred = pd.DataFrame({"date": predictions[code]["dates"]})
    mod_keys = [k for k in predictions[code] if k != "dates"]
    for k in mod_keys:
        df_pred[k] = predictions[code][k]
    df_pred["ym"] = df_pred["date"].dt.to_period("M")
    df_m = df_pred.groupby("ym")[mod_keys].mean().reset_index()
    df_m["date"] = df_m["ym"].dt.to_timestamp()
    
    st_obs = df_obs[df_obs["station_code"] == code]
    m = pd.merge(df_m, st_obs[["date", "q_obs_m3s"]], on="date", how="inner")
    
    ax.plot(m["date"], m["q_obs_m3s"], label="Observé (Vérité Terrain)", color="black", lw=2, zorder=5)
    if "temporal_model" in m.columns:
        ax.plot(m["date"], m["temporal_model"], label="Modèle Temporel (247 bassins)", color="#2563EB", lw=1.5, alpha=0.85)
    if "ensemble_3seeds" in m.columns:
        ax.plot(m["date"], m["ensemble_3seeds"], label="Ensemble Déploiement (3 graines)", color="#DC2626", lw=1.5, ls="--")
    elif "deploy_s42" in m.columns:
        ax.plot(m["date"], m["deploy_s42"], label="Déploiement Graine 42", color="#DC2626", lw=1.5, ls="--")
        
    ax.set_title(f"[{code}] {st_meta['name']} ({st_meta['basin']}) — Aire : {st_meta['delineated_area_km2']:.1f} km²", fontsize=12, fontweight="bold")
    ax.set_ylabel("Débit mensuel (m³/s)", fontsize=10)
    ax.grid(True, alpha=0.3, ls=":")
    ax.legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig("hydrographs_new_stations_comparison.png", dpi=200)
plt.show()
print("Figure sauvegardée : hydrographs_new_stations_comparison.png")
